# 4. LLM Inference

## Mô hình hỗ trợ
| Model | Provider | Backend |
|---|---|---|
| `claude-haiku-4-5` | Anthropic | REST API |
| `gpt-4.1-nano` | OpenAI | REST API |
| `gemini-2.5-flash-lite` | Google Vertex AI | ADC |
| `qwen3.5:9b` | Local | Ollama (GPU) |
| `llama3.1:8b` | Local | Ollama (GPU) |

## Prompt strategies
- **Zero-shot** : chỉ mô tả task + output format
- **Few-shot** : 2 ví dụ minh hoạ input→output trước khi hỏi
- **CoT** : yêu cầu model suy luận từng bước trước khi ra JSON

## Cài đặt thư viện và Repo

!pip install openai anthropic google-genai requests tqdm ollama

In [ ]:
!git clone https://github.com/DuongThai2712/smart-tourism

## Cấu hình API keys & tham số toàn cục

In [ ]:
import os
import torch

#  API keys (ưu tiên biến môi trường, fallback điền trực tiếp) ─
# os.environ["OPENAI_API_KEY"]    = "sk-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

OPENAI_KEY    = os.getenv("OPENAI_API_KEY",    "")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY", "")

#  Google Vertex AI — dùng ADC (Application Default Credentials) ─
# Không cần API key; chạy `gcloud auth application-default login` trước
VERTEX_PROJECT  = "gemini-498219"
VERTEX_LOCATION = "global"

#  Ollama local 
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

#  GPU detection ─
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_MEM  = torch.cuda.get_device_properties(0).total_memory // (1024**3)
    print(f"GPU: {GPU_NAME} ({GPU_MEM} GB VRAM)")
else:
    print("GPU: không có → Ollama dùng CPU")

#  Tham số inference
MAX_TOKENS = 1024   # tăng lên cho CoT
TEMPERATURE = 0.2
BATCH_SIZE = 8      # số context xử lý song song mỗi lần
MAX_RETRIES = 3
DELAY_BETWEEN_CALLS = 0.8   # giây giữa các API call (tránh rate limit)

PROMPT_STRATEGIES = ["zero_shot", "few_shot", "cot"]

MODELS_CONFIG = {
    "claude-haiku-4-5":      {"backend": "anthropic",  "enabled": bool(ANTHROPIC_KEY)},
    "gpt-4.1-nano":          {"backend": "openai",     "enabled": bool(OPENAI_KEY)},
    "gemini-3.1-flash-lite": {"backend": "vertex",     "enabled": True},
    "qwen3.5:9b":            {"backend": "ollama",     "enabled": True},
    "llama3.1:8b":           {"backend": "ollama",     "enabled": True},
}

print("\nTrạng thái models:")
for m, cfg in MODELS_CONFIG.items():
    status = "✓ enabled" if cfg["enabled"] else "✗ disabled (thiếu key)"
    print(f"  [{cfg['backend']:12s}] {m:30s} {status}")

## Load context

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

project_root = Path("/content/smart-tourism")

rag_path    = project_root / "Datasets" / "RAG"
raw_path    = project_root / "Datasets" / "Raw"
result_path = project_root / "Datasets" / "Results"
result_path.mkdir(parents=True, exist_ok=True)


In [ ]:
with open(rag_path / "llm_contexts.json", encoding="utf-8") as f:
    all_contexts = json.load(f)

print(f"Đã load {len(all_contexts)} contexts")
print(f"Cities : {sorted(set(c['city'] for c in all_contexts))}")

ctx0 = all_contexts[0]
print(f"\nVí dụ context đầu tiên:")
print(f"  user_id         :{ctx0['user_id']}")
print(f"  city            : {ctx0['city']}")
print(f"  budget_minutes  : {ctx0['budget_minutes']}")
print(f"  retrieved_pois  : {len(ctx0['retrieved_pois'])} POI")

## Prompt Strategies

Mỗi strategy gồm `system_prompt` + hàm `build_user_prompt(ctx)`.  
Tất cả đều yêu cầu output JSON cùng schema để parse thống nhất.

In [ ]:
OUTPUT_SCHEMA = """\
Return ONLY a valid JSON object with this exact schema (no markdown, no extra text):
{"itinerary": ["POI Name 1", "POI Name 2", ...], "total_time_min": <integer>, "reason": "<brief>"}"""


def _profile_block(ctx: dict) -> str:
    """Shared tourist profile block."""
    visited  = ctx["user_profile"]["visited_pois"]
    top_cats = ctx["user_profile"]["top_categories"]
    visited_str = ", ".join(visited) if visited else "None yet"
    cat_lines = "\n".join(
        f"  - {c['poiTheme']}: interest {c['int_u_c']:.2f}"
        for c in top_cats
    ) if top_cats else "  - No strong preferences detected"
    return (
        f"City: {ctx['city']}\n"
        f"Time budget: {ctx['budget_minutes']} minutes\n"
        f"Previously visited: {visited_str}\n"
        f"Interest categories:\n{cat_lines}"
    )


def _poi_block(ctx: dict) -> str:
    """Numbered POI list block."""
    return "\n".join(
        f"  {i+1:2d}. [{p['category']:12s}] {p['name']:35s} "
        f"~{p['avg_visit_min']:3.0f}min  sim={p['similarity']:.3f}"
        for i, p in enumerate(ctx["retrieved_pois"])
    )



In [ ]:
# STRATEGY 1 — ZERO-SHOT
ZERO_SHOT_SYSTEM = f"""\
You are an expert one-day tourism itinerary planner.
Rules:
1. Select 3-6 POIs that fit within the tourist's time budget.
2. Only use POIs from the PROVIDED LIST.
3. Order POIs logically (by proximity or thematic flow).
4. Respect each POI's average visit duration.
{OUTPUT_SCHEMA}"""


def zero_shot_user(ctx: dict) -> str:
    return (
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Available POIs:\n{_poi_block(ctx)}\n\n"
        f"Generate the optimal itinerary."
    )



# STRATEGY 2 — FEW-SHOT (2 examples)
FEW_SHOT_EXAMPLES = """\
=== EXAMPLE 1 ===
Tourist profile:
  City: Athens | Budget: 300 min | Interests: Museum (3.2), Historical (2.8)
Available POIs:
  1. [Museum     ] Acropolis Museum          ~90min
  2. [Historical ] Parthenon                 ~60min
  3. [Shopping   ] Monastiraki Flea Market   ~45min
  4. [Park       ] National Garden           ~30min
  5. [Historical ] Temple of Olympian Zeus   ~40min
Output:
{"itinerary": ["Parthenon", "Acropolis Museum", "Temple of Olympian Zeus", "Monastiraki Flea Market"], "total_time_min": 255, "reason": "Cluster historical sites first, then museum nearby, end at market."}

=== EXAMPLE 2 ===
Tourist profile:
  City: Barcelona | Budget: 240 min | Interests: Architecture (4.1), Art (2.5)
Available POIs:
  1. [Architecture] Sagrada Familia          ~90min
  2. [Art         ] Picasso Museum           ~60min
  3. [Architecture] Casa Batllo              ~45min
  4. [Beach       ] Barceloneta Beach        ~60min
  5. [Park        ] Park Guell               ~50min
Output:
{"itinerary": ["Sagrada Familia", "Casa Batllo", "Picasso Museum"], "total_time_min": 195, "reason": "Architecture focus, grouped on Eixample then Gothic Quarter."}

=== YOUR TASK ==="""

FEW_SHOT_SYSTEM = f"""\
You are an expert one-day tourism itinerary planner.
Rules:
1. Select 3-6 POIs that fit within the tourist's time budget.
2. Only use POIs from the PROVIDED LIST.
3. Order POIs logically (by proximity or thematic flow).
4. Respect each POI's average visit duration.
{OUTPUT_SCHEMA}"""


def few_shot_user(ctx: dict) -> str:
    return (
        f"{FEW_SHOT_EXAMPLES}\n"
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Available POIs:\n{_poi_block(ctx)}\n\n"
        f"Output:"
    )



# STRATEGY 3 — CHAIN-OF-THOUGHT (CoT)
COT_SYSTEM = f"""\
You are an expert one-day tourism itinerary planner.
Think step by step before producing the final itinerary:
  Step 1 — Identify the tourist's top interest categories.
  Step 2 — Filter POIs matching those categories from the list.
  Step 3 — Check time budget: sum(visit_duration) + estimated travel must stay within budget.
  Step 4 — Order selected POIs by geographic proximity or logical theme flow.
  Step 5 — Output the final JSON.
Rules:
- Select 3-6 POIs only from the PROVIDED LIST.
- Respect each POI's average visit duration.
Write your reasoning in a <thinking> block, then output the JSON.
{OUTPUT_SCHEMA}"""


def cot_user(ctx: dict) -> str:
    return (
        f"Tourist profile:\n{_profile_block(ctx)}\n\n"
        f"Available POIs (name | category | avg_visit_min | relevance_score):\n"
        f"{_poi_block(ctx)}\n\n"
        f"Think step by step, then output the JSON."
    )



In [ ]:
PROMPT_REGISTRY = {
    "zero_shot": {"system": ZERO_SHOT_SYSTEM, "user_fn": zero_shot_user},
    "few_shot":  {"system": FEW_SHOT_SYSTEM,  "user_fn": few_shot_user},
    "cot":       {"system": COT_SYSTEM,        "user_fn": cot_user},
}

print("3 prompt strategies đã sẵn sàng: zero_shot | few_shot | cot")

#  Kiểm tra nhanh 
for strat, reg in PROMPT_REGISTRY.items():
    user_msg = reg["user_fn"](ctx0)
    print(f"\n[{strat}] system={len(reg['system'])} chars | user_msg={len(user_msg)} chars")
    print(f"  User preview: {user_msg[:120].strip()} ...")

## Response Parser

Parse JSON từ response LLM, xử lý các trường hợp:
- Markdown fences ```json ... ```
- <thinking>...</thinking> tags (CoT / extended thinking)
- Text thừa trước/sau JSON

In [ ]:
import re

def parse_llm_response(raw: str) -> dict:
    if not raw:
        return {"itinerary": [], "total_time_min": 0, "reason": "empty_response"}

    # Loại bỏ thinking tags (CoT responses)
    text = re.sub(r"<thinking>.*?</thinking>", "", raw, flags=re.DOTALL)
    # Loại bỏ markdown fences
    text = re.sub(r"```(?:json)?\s*", "", text)
    text = text.strip()

    # Tìm JSON object đầu tiên
    match = re.search(r"\{[^{}]*\}", text, re.DOTALL)
    if not match:
        # Thử tìm JSON lồng nhau
        match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            parsed = json.loads(match.group())
            # Đảm bảo có đủ các key cần thiết
            parsed.setdefault("itinerary", [])
            parsed.setdefault("total_time_min", 0)
            parsed.setdefault("reason", "")
            # Đảm bảo itinerary là list of strings
            if not isinstance(parsed["itinerary"], list):
                parsed["itinerary"] = []
            return parsed
        except json.JSONDecodeError:
            pass

    return {
        "itinerary": [],
        "total_time_min": 0,
        "reason": "parse_error",
        "raw_truncated": raw[:300],
    }

In [ ]:
# Test parser
test_cases = [
    '{"itinerary": ["A", "B"], "total_time_min": 120, "reason": "ok"}',
    '```json\n{"itinerary": ["X"], "total_time_min": 60, "reason": "ok"}\n```',
    '<thinking>Step 1: check budget</thinking>\n{"itinerary": ["Y"], "total_time_min": 90, "reason": "cot"}',
    'Here is the result: {"itinerary": ["Z"], "total_time_min": 45, "reason": "extra text"}',
    'invalid response without json',
]
for tc in test_cases:
    result = parse_llm_response(tc)
    status = "✓" if result["itinerary"] else "✗"
    print(f"{status} itinerary={result['itinerary']} | reason={result['reason']}")

## LLM Backend Wrappers

In [ ]:
import time
import requests


def _make_result(model, strategy, raw_out, latency, prompt_tok, completion_tok):
    """Tạo dict kết quả chuẩn từ raw LLM output."""
    parsed = parse_llm_response(raw_out)
    parsed.update({
        "model":             model,
        "prompt_strategy":   strategy,
        "latency_sec":       round(latency, 3),
        "prompt_tokens":     prompt_tok,
        "completion_tokens": completion_tok,
    })
    return parsed


#  1. Anthropic — Claude Haiku 4.5 
def call_anthropic(model_id: str, system: str, user_msg: str,
                   strategy: str) -> dict:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
    t0 = time.time()
    resp = client.messages.create(
        model      = model_id,
        max_tokens = MAX_TOKENS,
        temperature= TEMPERATURE,
        system     = system,
        messages   = [{"role": "user", "content": user_msg}],
    )
    return _make_result(
        model_id, strategy,
        raw_out        = resp.content[0].text,
        latency        = time.time() - t0,
        prompt_tok     = resp.usage.input_tokens,
        completion_tok = resp.usage.output_tokens,
    )


#  2. OpenAI — GPT-5.4-nano 
def call_openai(model_id: str, system: str, user_msg: str,
                strategy: str) -> dict:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_KEY)
    t0 = time.time()
    resp = client.chat.completions.create(
        model       = model_id,
        max_tokens  = MAX_TOKENS,
        temperature = TEMPERATURE,
        messages    = [
            {"role": "system", "content": system},
            {"role": "user",   "content": user_msg},
        ],
    )
    return _make_result(
        model_id, strategy,
        raw_out        = resp.choices[0].message.content,
        latency        = time.time() - t0,
        prompt_tok     = resp.usage.prompt_tokens,
        completion_tok = resp.usage.completion_tokens,
    )


#  3. Google Vertex AI — Gemini 3.1 Flash Lite (ADC) 
_vertex_client = None

def _get_vertex_client():
    global _vertex_client
    if _vertex_client is None:
        from google import genai
        _vertex_client = genai.Client(
            vertexai=True,
            project=VERTEX_PROJECT,
            location=VERTEX_LOCATION,
        )
    return _vertex_client


def call_vertex(model_id: str, system: str, user_msg: str,
                strategy: str) -> dict:
    from google.genai import types as gtypes
    client = _get_vertex_client()
    t0 = time.time()
    resp = client.models.generate_content(
        model    = model_id,
        contents = user_msg,
        config   = gtypes.GenerateContentConfig(
            system_instruction = system,
            max_output_tokens  = MAX_TOKENS,
            temperature        = TEMPERATURE,
        ),
    )
    usage = resp.usage_metadata
    return _make_result(
        model_id, strategy,
        raw_out        = resp.text,
        latency        = time.time() - t0,
        prompt_tok     = getattr(usage, "prompt_token_count",     0) or 0,
        completion_tok = getattr(usage, "candidates_token_count", 0) or 0,
    )


#  4. Ollama — local models (qwen3.5:9b / llama3.1:8b) 
def _ollama_available() -> bool:
    """Kiểm tra Ollama server có đang chạy không."""
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=3)
        return r.status_code == 200
    except Exception:
        return False


def call_ollama(model_id: str, system: str, user_msg: str,
                strategy: str) -> dict:
    """
    Gọi Ollama REST API (/api/chat).
    Ollama tự detect GPU nếu model đã được pull và CUDA/ROCm khả dụng.
    Dùng num_gpu=-1 để load toàn bộ layers lên GPU.
    """
    payload = {
        "model": model_id,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": MAX_TOKENS,
            "num_gpu": -1 if DEVICE == "cuda" else 0,  # -1 = all layers on GPU
        },
        "messages": [
            {"role": "system",    "content": system},
            {"role": "user",      "content": user_msg},
        ],
    }
    t0 = time.time()
    resp = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json=payload,
        timeout=120,
    )
    resp.raise_for_status()
    data = resp.json()
    raw_text   = data.get("message", {}).get("content", "")
    usage      = data.get("prompt_eval_count", 0)
    completion = data.get("eval_count", 0)
    return _make_result(
        model_id, strategy,
        raw_out        = raw_text,
        latency        = time.time() - t0,
        prompt_tok     = usage,
        completion_tok = completion,
    )


In [ ]:
#  Dispatch table 
BACKEND_FN = {
    "anthropic": call_anthropic,
    "openai":    call_openai,
    "vertex":    call_vertex,
    "ollama":    call_ollama,
}


def call_model(model_name: str, strategy: str, ctx: dict) -> dict:
    """
    Gọi model với strategy chỉ định, có retry + exponential backoff.
    Trả về dict kết quả chuẩn.
    """
    cfg      = MODELS_CONFIG[model_name]
    backend  = cfg["backend"]
    reg      = PROMPT_REGISTRY[strategy]
    system   = reg["system"]
    user_msg = reg["user_fn"](ctx)
    fn       = BACKEND_FN[backend]

    for attempt in range(MAX_RETRIES):
        try:
            result = fn(model_name, system, user_msg, strategy)
            result["user_id"] = ctx["user_id"]
            result["city"]    = ctx["city"]
            return result
        except Exception as e:
            wait = 2 ** attempt
            print(f"    [{model_name}/{strategy}] attempt {attempt+1} failed: {e} — retry in {wait}s")
            if attempt < MAX_RETRIES - 1:
                time.sleep(wait)

    # Trả về empty result sau khi hết retry
    return {
        "model": model_name, "prompt_strategy": strategy,
        "user_id": ctx["user_id"], "city": ctx["city"],
        "itinerary": [], "total_time_min": 0, "reason": "max_retries_exceeded",
        "latency_sec": 0, "prompt_tokens": 0, "completion_tokens": 0,
    }


#  Kiểm tra Ollama ─
ollama_ok = _ollama_available()
print(f"Ollama server ({OLLAMA_BASE_URL}): {'✓ online' if ollama_ok else '✗ offline'}")
if not ollama_ok:
    print("  → Các model ollama sẽ bị skip. Chạy: ollama serve")
    for m, cfg in MODELS_CONFIG.items():
        if cfg["backend"] == "ollama":
            cfg["enabled"] = False

print("\nBackend wrappers đã sẵn sàng.")

## Metrics

In [ ]:
def normalize_poi(name: str) -> str:
    return name.strip().lower().replace(" ", "_").replace("-", "_")


def compute_itinerary_metrics(predicted: list, actual: list) -> dict:
    """Recall, Precision, F-score (Section 7.2 bài báo)."""
    pred_set   = set(normalize_poi(p) for p in predicted)
    actual_set = set(normalize_poi(a) for a in actual)
    if not actual_set or not pred_set:
        return {"recall": 0.0, "precision": 0.0, "f_score": 0.0}
    tp        = len(pred_set & actual_set)
    recall    = tp / len(actual_set)
    precision = tp / len(pred_set)
    f_score   = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {
        "recall":    round(recall,    4),
        "precision": round(precision, 4),
        "f_score":   round(f_score,   4),
    }


def compute_ae_ue(itinerary: list, profit_per_poi: dict, max_profit: float = 1.0) -> dict:
    """AE, UE (Eq. 22, 23 bài báo — simplified for LLM output)."""
    if not itinerary:
        return {"ae": 0.0, "ue": 0.0}
    profits = [profit_per_poi.get(normalize_poi(p), 0.0) for p in itinerary]
    ae = sum(profits) / (len(profits) * max_profit) if max_profit > 0 else 0.0
    ue = ae * (sum(profits) / max(max_profit * len(profits), 1e-9))
    return {"ae": round(ae, 4), "ue": round(ue, 4)}


print("Metrics functions ready.")

## Load Ground Truth & Profit Map

In [ ]:
#  Load visit và POI data ─
visit_frames, poi_frames = [], []
for city_dir in sorted(raw_path.iterdir()):
    if not city_dir.is_dir():
        continue
    city = city_dir.name
    for fname, store in [("touristsVisits.csv", visit_frames), ("POIs.csv", poi_frames)]:
        fp = city_dir / fname
        if fp.exists():
            df = pd.read_csv(fp)
            df["city"] = city
            store.append(df)

visit_pdf = pd.concat(visit_frames, ignore_index=True)
poi_pdf   = pd.concat(poi_frames,   ignore_index=True)

#  Profit proxy = normalized visit count ─
pop = (
    visit_pdf.groupby(["city", "poiID"])["userID"].count()
    .reset_index(name="pop")
    .merge(poi_pdf[["city", "poiID", "poiName"]], on=["city", "poiID"])
)
city_max    = pop.groupby("city")["pop"].max().rename("pop_max")
pop         = pop.join(city_max, on="city")
pop["profit_norm"] = pop["pop"] / pop["pop_max"]

# Dict (city, poi_normalized) → profit
profit_map = {
    (r["city"], normalize_poi(r["poiName"])): r["profit_norm"]
    for _, r in pop.iterrows()
}

# Dict user_id → actual sequence (sorted by timestamp)
actual_map = (
    visit_pdf
    .merge(poi_pdf[["city", "poiID", "poiName"]], on=["city", "poiID"])
    .sort_values("unixTimestamp")
    .groupby(["userID", "city"])["poiName"]
    .apply(lambda x: list(dict.fromkeys(x)))  # deduplicate giữ thứ tự
    .to_dict()
)

print(f"Profit map : {len(profit_map):,} entries")
print(f"Actual map : {len(actual_map):,} (user, city) pairs")

## Batch Inference Engine

Chạy inference cho 1 (model, strategy, context) tuple.
    Tính metrics ngay sau khi nhận kết quả.

Batch inference:
- Local models (ollama): ThreadPoolExecutor để chạy song song trong batch
- API models (anthropic/openai/vertex): tuần tự với delay tránh rate limit

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm


def run_single(model_name: str, strategy: str, ctx: dict,
               actual_map: dict, profit_map: dict) -> dict:

    result = call_model(model_name, strategy, ctx)
    itinerary = result.get("itinerary", [])
    city = ctx["city"]
    user_id = ctx["user_id"]

    actual_seq = actual_map.get((user_id, city), [])
    city_profit = {k[1]: v for k, v in profit_map.items() if k[0] == city}

    trad_m  = compute_itinerary_metrics(itinerary, actual_seq)
    alloc_m = compute_ae_ue(itinerary, city_profit)

    return {"user_id": user_id, "city": city,
        "budget_min": ctx["budget_minutes"],
        "model": model_name, "prompt_strategy": strategy,
        "predicted_itinerary":  "|".join(itinerary),
        "n_pois_predicted": len(itinerary),
        "actual_itinerary": "|".join(actual_seq),
        "n_pois_actual": len(actual_seq),
        "predicted_time_min": result.get("total_time_min", 0),
        "reason": result.get("reason", ""),
        "latency_sec": result.get("latency_sec", 0),
        "prompt_tokens": result.get("prompt_tokens", 0),
        "completion_tokens": result.get("completion_tokens", 0),
        **trad_m,
        **alloc_m,
    }




In [ ]:
def run_batch_inference(
    contexts: list,
    models_config: dict,
    strategies: list,
    actual_map: dict,
    profit_map: dict,
    batch_size: int = BATCH_SIZE,
    api_delay: float = DELAY_BETWEEN_CALLS,
) -> list:

    enabled_models = [m for m, c in models_config.items() if c["enabled"]]
    total_jobs     = len(contexts) * len(enabled_models) * len(strategies)

    print(f"Tổng jobs: {total_jobs:,} "
          f"({len(contexts)} ctx × {len(enabled_models)} models × {len(strategies)} strategies)")
    print(f"Models   : {enabled_models}")
    print(f"Strategies: {strategies}")

    all_results  = []
    pbar         = tqdm(total=total_jobs, desc="Inference", unit="job")

    # Nhóm contexts thành batches
    ctx_batches = [
        contexts[i : i + batch_size]
        for i in range(0, len(contexts), batch_size)
    ]

    for model_name in enabled_models:
        backend   = models_config[model_name]["backend"]
        is_local  = backend == "ollama"
        # Local models: parallelism = batch_size; API: parallelism = 1
        max_workers = batch_size if is_local else 1

        for strategy in strategies:
            print(f"\n {model_name} / {strategy} ")

            for batch_idx, batch in enumerate(ctx_batches):
                with ThreadPoolExecutor(max_workers=max_workers) as pool:
                    futures = {
                        pool.submit(
                            run_single,
                            model_name, strategy, ctx,
                            actual_map, profit_map
                        ): ctx
                        for ctx in batch
                    }
                    for future in as_completed(futures):
                        try:
                            row = future.result()
                            all_results.append(row)
                        except Exception as e:
                            print(f"  future error: {e}")
                        pbar.update(1)

                # Delay sau mỗi batch — chỉ áp dụng cho API calls
                if not is_local and batch_idx < len(ctx_batches) - 1:
                    time.sleep(api_delay * batch_size)

    pbar.close()
    print(f"\n✓ Hoàn thành: {len(all_results):,} / {total_jobs:,} jobs")
    return all_results


print("✓ Batch inference engine sẵn sàng.")

## Chạy Inference

In [ ]:
print(f"Số contexts sẽ chạy: {len(all_contexts)}")

all_results = run_batch_inference(
    contexts       = all_contexts,
    models_config  = MODELS_CONFIG,
    strategies     = PROMPT_STRATEGIES,
    actual_map     = actual_map,
    profit_map     = profit_map,
    batch_size     = BATCH_SIZE,
    api_delay      = DELAY_BETWEEN_CALLS,
)

## Lưu kết quả

In [ ]:
results_df = pd.DataFrame(all_results)

# Lưu raw
results_df.to_csv(result_path / "llm_results_v2.csv", index=False)
results_df.to_json(result_path / "llm_results_v2.json", orient="records", indent=2)

print(f"✓ Đã lưu {len(results_df):,} rows")
print(f"  Columns: {list(results_df.columns)}")
results_df.head(3)

In [ ]:
metrics_cols = ["recall", "precision", "f_score", "ae", "ue"]

#  Theo model × strategy 
summary_model_strategy = (
    results_df
    .groupby(["model", "prompt_strategy"])[metrics_cols]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(["model", "prompt_strategy"])
)

print("=" * 90)
print("BẢNG 1 — Kết quả trung bình theo Model × Prompt Strategy")
print("=" * 90)
print(summary_model_strategy.to_string(index=False))
summary_model_strategy.to_csv(result_path / "summary_model_strategy.csv", index=False)

In [ ]:
#  Theo model × city (strategy tốt nhất) 
best_strategy_per_model = (
    summary_model_strategy
    .sort_values("f_score", ascending=False)
    .groupby("model")
    .first()
    .reset_index()[["model", "prompt_strategy", "recall", "precision", "f_score", "ae", "ue"]]
)

print("\n" + "=" * 90)
print("BẢNG 2 — Best prompt strategy per model (theo F-score)")
print("=" * 90)
print(best_strategy_per_model.to_string(index=False))

#  Theo city × model (strategy = best f-score) 
summary_city = (
    results_df
    .groupby(["city", "model"])[metrics_cols]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(["city", "f_score"], ascending=[True, False])
)


print("BẢNG 3 — Kết quả theo City × Model (trung bình tất cả strategies)")
print("=" * 90)
print(summary_city.to_string(index=False))
summary_city.to_csv(result_path / "summary_city_model.csv", index=False)

In [ ]:
#  Chi phí token và latency ─
cost_df = (
    results_df
    .groupby(["model", "prompt_strategy"])[["latency_sec", "prompt_tokens", "completion_tokens"]]
    .mean()
    .round(2)
    .reset_index()
)
cost_df["total_tokens"] = cost_df["prompt_tokens"] + cost_df["completion_tokens"]

# Giá USD per 1M tokens (cập nhật tháng 6/2026)
PRICE = {
    "claude-haiku-4-5":      {"in": 0.80,  "out": 4.00},
    "gpt-4.1-nano":          {"in": 0.10,  "out": 0.40},
    "gemini-2.5-flash-lite": {"in": 0.075, "out": 0.30},
    "qwen3.5:9b":            {"in": 0.0,   "out": 0.0},   # local
    "llama3.1:8b":           {"in": 0.0,   "out": 0.0},   # local
}
cost_df["cost_usd_per_call"] = cost_df.apply(
    lambda r: round(
        r["prompt_tokens"]     / 1e6 * PRICE.get(r["model"], {"in":0})["in"] +
        r["completion_tokens"] / 1e6 * PRICE.get(r["model"], {"out":0})["out"],
        6
    ), axis=1
)


print("BẢNG 4 — Chi phí token & latency trung bình")
print("=" * 90)
print(cost_df.to_string(index=False))
cost_df.to_csv(result_path / "cost_latency_summary.csv", index=False)

print("\nTất cả output đã lưu:")
for f in sorted(result_path.iterdir()):
    size_kb = f.stat().st_size // 1024
    print(f"  {f.name:45s} {size_kb:6d} KB")

In [ ]:
# Tổng hợp score = 0.4*f_score + 0.3*ae + 0.3*ue
ranking = summary_model_strategy.copy()
ranking["composite_score"] = (
    0.4 * ranking["f_score"] +
    0.3 * ranking["ae"] +
    0.3 * ranking["ue"]
).round(4)

print("=" * 60)
print("RANKING TỔNG HỢP (0.4×F1 + 0.3×AE + 0.3×UE)")
print("=" * 60)
print(
    ranking
    .sort_values("composite_score", ascending=False)
    [["model", "prompt_strategy", "f_score", "ae", "ue", "composite_score"]]
    .to_string(index=False)
)

ranking.to_csv(result_path / "final_ranking.csv", index=False)

## Push lên Git

In [ ]:
%cd /content/smart-tourism

In [ ]:
!git remote -v
!git status

In [ ]:
!git add Datasets/Results

In [ ]:
!git pull origin main

In [ ]:
!git commit -m "Add results"

In [ ]:
!git push origin main